# MediaForge on a free GPU (Google Colab)

Runs the **whole MediaForge app on a free Colab T4 GPU (16 GB)** and gives you a
link to open it in your browser. Motion generation (LTX-Video / Wan) runs on the
GPU for **free** — no API keys, no ngrok account.

**Before running: Runtime → Change runtime type → Hardware accelerator → GPU → Save.**

Then run each cell in order with Shift+Enter. The last cell prints a link — click it.

## 1 · Confirm the GPU is on

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Do Runtime > Change runtime type > GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))

## 2 · Get MediaForge + install dependencies (~2 min)

In [ ]:
import os
if not os.path.isdir('/content/mediaforge'):
    !git clone -q https://github.com/bigmo6286/mediaforge.git /content/mediaforge
%cd /content/mediaforge/backend
# Torch is already installed on Colab with CUDA; add the rest.
!pip install -q fastapi 'uvicorn[standard]' python-multipart httpx imageio-ffmpeg Pillow
!pip install -q diffusers transformers accelerate sentencepiece
print('\nInstalled.')

## 3 · Run everything locally on the GPU (no keys)

In [ ]:
import os
os.environ['MOTION_MODEL'] = 'ltx'        # efficient video model
os.environ['MOTION_PROVIDER'] = 'local'   # run on this GPU
os.environ['WAN_PROVIDER'] = 'local'
print('Configured for local GPU generation.')

## 4 · Start MediaForge and open it
Run this, wait ~10 seconds, then click the printed link.

In [ ]:
import subprocess, time, sys
# Start the server in the background (serves the UI + API on port 8000).
proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app',
                         '--host', '0.0.0.0', '--port', '8000'],
                        cwd='/content/mediaforge/backend')
time.sleep(10)
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
print('\n==============================================')
print(' Open MediaForge here:')
print(' ', url)
print('==============================================')
print('\nGo to the Motion tab, type a prompt, and Generate.')
print('First generation downloads the model weights (a few minutes).')
print('Keep this Colab tab open while you use the app.')

## Notes
* **Motion (text/image → video)** runs on the free GPU out of the box.
* **Talking Avatar** needs SadTalker set up separately; the easiest route for
  avatars is a small hosted credit (add a fal key in the app's Settings tab).
* Free Colab sessions time out after a while and the GPU may be busy at peak
  times — just rerun the cells for a fresh session.
* To stop: Runtime → Disconnect and delete runtime.